In [3]:
import os
import pandas as pd
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    SplineTransformer,
    QuantileTransformer,
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)
from dotenv import load_dotenv

load_dotenv()

TABLE_NAME = "users_churn"  # таблица с данными в postgres

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5003
TRACKING_SERVER_URL = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

EXPERIMENT_NAME = "PREPROCESSING_EXPERIMENT"
RUN_NAME = "preprocessing_run_1"
REGISTRY_MODEL_NAME = "your_model_name"
ASSETS_DIR = "assets"

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(TRACKING_SERVER_URL)
mlflow.set_registry_uri(TRACKING_SERVER_URL)

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment_id:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME)


# with mlflow.start_run(
#     run_name=RUN_NAME, experiment_id=experiment_id.experiment_id
# ) as run:
#     run_id = run.info.run_id

#     # mlflow.log_artifacts(ASSETS_DIR)

In [17]:
import os
from dotenv import load_dotenv
import psycopg
import pandas as pd

load_dotenv()

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:
    # напишите код здесь
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM users_churn")
        data = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

df = pd.DataFrame(data, columns=columns)

In [5]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# определение категориальных колонок, которые будут преобразованы
cat_columns = ["type", "payment_method", "internet_service", "gender"]

# создание объекта OneHotEncoder для преобразования категориальных переменных
# auto - автоматическое определение категорий
# ignore - игнорировать ошибки, если встречается неизвестная категория
# max_categories - максимальное количество уникальных категорий
# sparse_output - вывод в виде разреженной матрицы, если False, то в виде обычного массива
# drop="first" - удаляет первую категорию, чтобы избежать ловушки мультиколлинеарности
encoder_oh = OneHotEncoder(
    categories="auto",
    handle_unknown="ignore",
    max_categories=10,
    sparse_output=False,
    drop="first",
)

# применение OneHotEncoder к данным. Преобразование категориальных данных в массив
encoded_features = encoder_oh.fit_transform(df[cat_columns].to_numpy())

# преобразование полученных признаков в DataFrame и установка названий колонок
# get_feature_names_out() - получение имён признаков после преобразования
encoded_df = pd.DataFrame(
    encoded_features, columns=encoder_oh.get_feature_names_out(cat_columns)
)

# конкатенация исходного DataFrame с новым DataFrame, содержащим закодированные категориальные признаки
# axis=1 означает конкатенацию по колонкам
df1 = pd.concat([df, encoded_df], axis=1)

df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,1,7590-VHVEG,2020-01-01,NaT,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,...,No,No,No,No,Female,0,Yes,No,None,0
1,2,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,...,Yes,No,No,No,Male,0,No,No,No,0


In [ ]:
from sklearn.preprocessing import (
    SplineTransformer,
    QuantileTransformer,
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_columns = ["monthly_charges", "total_charges"]

n_knots = 3
degree_spline = 4
n_quantiles = 100
degree = 3
n_bins = 5
encode = "ordinal"
strategy = "uniform"
subsample = None


# SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)
encoded_features = encoder_spl.fit_transform(
    df[num_columns].fillna(0).to_numpy()
)

encoded_df = pd.DataFrame(
    encoded_features, columns=encoder_spl.get_feature_names_out(num_columns)
)
num_df = pd.concat([df, encoded_df], axis=1)


# QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)
encoded_features = encoder_q.fit_transform(
    df[num_columns].fillna(0).to_numpy()
)  # замени другие по образцу этому

encoded_df = pd.DataFrame(
    encoded_features, columns=encoder_q.get_feature_names_out(num_columns)
)
encoded_df.columns = [col + f"_q_{n_quantiles}" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(
    df[num_columns].fillna(0).to_numpy()
)

encoded_df = pd.DataFrame(
    encoded_features, columns=encoder_rb.get_feature_names_out(num_columns)
)
encoded_df.columns = [col + f"_robust" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(
    df[num_columns].fillna(0).to_numpy()
)

encoded_df = pd.DataFrame(
    encoded_features, columns=encoder_pol.get_feature_names_out(num_columns)
)
# get all columns after the intercept and original features
encoded_df = encoded_df.iloc[:, 1 + len(num_columns) :]
encoded_df.columns = [f"{col}_poly" for col in encoded_df.columns]
num_df = pd.concat([num_df, encoded_df], axis=1)

# KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(
    n_bins=n_bins, encode=encode, strategy=strategy, subsample=subsample
)
encoded_features = encoder_kbd.fit_transform(
    df[num_columns].fillna(0).to_numpy()
)

encoded_df = pd.DataFrame(
    encoded_features, columns=encoder_kbd.get_feature_names_out(num_columns)
)
encoded_df.columns = [col + f"_bin" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


num_df.head(2)

numeric_transformer = ColumnTransformer(
    transformers=[
        ("spl", encoder_spl, num_columns),
        ("q", encoder_q, num_columns),
        ("rb", encoder_rb, num_columns),
        ("pol", encoder_pol, num_columns),
        ("kbd", encoder_kbd, num_columns),
    ]
)

categorical_transformer = Pipeline(steps=[("encoder", encoder_oh)])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_columns),
        ("cat", categorical_transformer, cat_columns),
    ],
    n_jobs=-1,
)

df1 = df.copy()
df1[num_columns] = df1[num_columns].fillna(0)
encoded_features = preprocessor.fit_transform(df1)  # ваш код здесь #

transformed_df = pd.DataFrame(
    encoded_features, columns=preprocessor.get_feature_names_out()
)  # ваш код здесь #

# # добавь и удаление исходных колонок, чтобы не было дублирования
df1 = pd.concat([df1, transformed_df], axis=1)
df1.drop(columns=num_columns + cat_columns, inplace=True)
df1.head(2)

,id,customer_id,begin_date,end_date,paperless_billing,online_security,online_backup,device_protection,tech_support,streaming_tv,...,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__internet_service_None,cat__gender_Male
0,1,7590-VHVEG,2020-01-01,NaT,Yes,No,Yes,No,No,No,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,5575-GNVDE,2017-04-01,NaT,No,Yes,No,Yes,No,No,...,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(auto_class_weights=auto_class_weights)

In [7]:
from sklearn import set_config

# Включаем текстово-графическую (HTML) визуализацию
set_config(display="diagram")

# Теперь, если просто вызвать пайплайн в конце ячейки Jupyter/Interactive Window:
preprocessor

ColumnTransformer(n_jobs=-1,
                  transformers=[('num',
                                 ColumnTransformer(transformers=[('spl',
                                                                  SplineTransformer(degree=4,
                                                                                    n_knots=3),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('q',
                                                                  QuantileTransformer(n_quantiles=100),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('rb',
                                                                  RobustScaler(),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('pol',
                                                                  PolynomialFeatures(degree=3),
                                                                  ['monthly_char...
                                                                   'total_charges']),
                                                                 ('kbd',
                                                                  KBinsDiscretizer(encode='ordinal',
                                                                                   strategy='uniform',
                                                                                   subsample=None),
                                                                  ['monthly_charges',
                                                                   'total_charges'])]),
                                 ['monthly_charges', 'total_charges']),
                                ('cat',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                max_categories=10,
                                                                sparse_output=False))]),
                                 ['type', 'payment_method', 'internet_service',
                                  'gender'])])

In [8]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(TRACKING_SERVER_URL)
mlflow.set_registry_uri(TRACKING_SERVER_URL)


RUN_NAME = "preprocessing_run_8"

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
if not experiment_id:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    experiment_id = mlflow.get_experiment_by_name(
        EXPERIMENT_NAME
    ).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    # set skops_trusted_types, set cloudpickle for serialization
    mlflow.sklearn.log_model(
        preprocessor,
        artifact_path="column_transformer",
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

2026/07/20 13:30:59 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/07/20 13:31:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run preprocessing_run_8 at: http://127.0.0.1:5003/#/experiments/13/runs/c439c3320a304fa8a426cf42f4e50e21
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/13


In [1]:
import mlflow

print(mlflow.__version__)

2.22.5
